# Streamlined KVG SNV/Indel Filter Pipeline

This notebook consolidates evaluation, simple GQ/DP thresholding, and XGBoost filtering from `kvg_filter_snv_indels.ipynb`.  
**Train** on TRAIN_SAMPLES, **evaluate** (threshold choice + metrics) on TEST_SAMPLES; **EVAL_SAMPLES** are held out for future Hiphase physical-phasing verification.

In [ ]:
# Pip installs (required for Terra VM) - exact versions per plan
!pip install numpy==1.26.4 scipy==1.15.3 xgboost==1.7.6 pysam pandas scikit-learn matplotlib plotly tqdm --quiet

In [ ]:
import numpy, scipy, xgboost
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("xgboost:", xgboost.__version__)

In [ ]:
# Build bcftools and install tabix, bgzip (required for Terra VM)
import os
import shutil
BIN_DIR = os.path.expanduser("~/.local/bin")
os.makedirs(BIN_DIR, exist_ok=True)
os.environ["PATH"] = BIN_DIR + os.pathsep + os.environ.get("PATH", "")
if not os.path.exists(os.path.join(BIN_DIR, "bcftools")):
    !rm -rf htslib bcftools 2>/dev/null; git clone --recurse-submodules https://github.com/samtools/htslib.git
    !git clone https://github.com/samtools/bcftools.git
    !cd bcftools && make
    !cp bcftools/bcftools "$BIN_DIR/"
if not os.path.exists(os.path.join(BIN_DIR, "tabix")):
    !wget -q -O htslib-1.23.tar.bz2 https://github.com/samtools/htslib/releases/download/1.23/htslib-1.23.tar.bz2
    !bunzip2 -c htslib-1.23.tar.bz2 | tar xf -
    !cd htslib-1.23 && ./configure --prefix="$BIN_DIR" && make && make install
    !cp htslib-1.23/tabix "$BIN_DIR/" 2>/dev/null || true
    !cp htslib-1.23/bgzip "$BIN_DIR/" 2>/dev/null || true
print("bcftools:", shutil.which("bcftools") or "not found")
print("tabix:", shutil.which("tabix") or "not found")
print("bgzip:", shutil.which("bgzip") or "not found")

## 2. Config (hardcoded paths, regions, three-way split)

In [ ]:
import os
import subprocess
import random
import pandas as pd
import numpy as np
import pysam
from collections import defaultdict

# --- Paths (hardcoded) ---
LOCAL_DIR = os.getcwd()
REF_FASTA = "GCA_000001405.15_GRCh38_no_alt_analysis_set.fa"
HPRC_METADATA_URL = "https://raw.githubusercontent.com/human-pangenomics/hprc_intermediate_assembly/main/data_tables/sample/hprc_release2_sample_metadata.csv"

# gVCFs (per-chrom subset)
GVCF_GCS_PATHS = [
    "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scratch/kvg/subset_chr20.g.vcf.bgz",
    "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scratch/kvg/subset_chr21.g.vcf.bgz",
    "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scratch/kvg/subset_chr22.g.vcf.bgz",
]

# Evaluation regions: extract these from gVCFs before normalizing (saves time)
LOCUS_START = 10_000_000
LOCUS_END = 11_000_000
EVAL_CHROMOSOMES = ["chr20", "chr21", "chr22"]
EVAL_REGIONS = {chrom: (LOCUS_START, LOCUS_END) for chrom in EVAL_CHROMOSOMES}

# Constants
SUBSAMPLE_PER_STRATUM = 50_000
SUPERPOPS = ["AFR", "AMR", "EAS", "EUR", "SAS"]
RANDOM_SEED = 42

In [ ]:
# Load TRUTH_VCFS and TRUTH_BEDS from kvg_filter_snv_indels.ipynb (same repo) to avoid duplicating 230 entries
import json
_src_nb = os.path.join(LOCAL_DIR, "kvg_filter_snv_indels.ipynb")
if os.path.exists(_src_nb):
    with open(_src_nb) as f:
        _nb = json.load(f)
    for _c in _nb["cells"]:
        if _c.get("cell_type") != "code" or not isinstance(_c.get("source"), list):
            continue
        _src = "".join(_c["source"])
        if "truth_vcfs = {" in _src and "truth_beds" not in _src:
            exec(_src)
            break
    for _c in _nb["cells"]:
        if _c.get("cell_type") != "code" or not isinstance(_c.get("source"), list):
            continue
        _src = "".join(_c["source"])
        if "truth_beds = {" in _src:
            exec(_src)
            break
else:
    truth_vcfs = {}
    truth_beds = {}

# HPRC metadata and three-way split (train / test / eval). Eval is held out for future Hiphase verification.
hprc_df = pd.read_csv(HPRC_METADATA_URL)
hprc_clean = hprc_df[hprc_df["superpopulation"].notna() & hprc_df["sex"].notna()].copy()
random.seed(RANDOM_SEED)
# 60% train, 20% test, 20% eval (stratified by superpop × sex)
train_samples, test_samples, eval_samples = [], [], []
for pop in SUPERPOPS:
    pop_df = hprc_clean[hprc_clean["superpopulation"] == pop]
    for sex in ["male", "female"]:
        sex_df = pop_df[pop_df["sex"] == sex]
        available = sex_df["sample_id"].tolist()
        n = len(available)
        if n < 3:
            continue
        n_train = max(1, round(n * 0.6))
        n_test = max(1, round((n - n_train) * 0.5))
        n_eval = n - n_train - n_test
        shuffled = random.sample(available, n)
        train_samples.extend(shuffled[:n_train])
        test_samples.extend(shuffled[n_train : n_train + n_test])
        eval_samples.extend(shuffled[n_train + n_test :])
TRAIN_SAMPLES = set(train_samples)
TEST_SAMPLES = set(test_samples)
EVAL_SAMPLES = set(eval_samples)
LABELED_SAMPLES = TRAIN_SAMPLES | TEST_SAMPLES | EVAL_SAMPLES
print("Train:", len(TRAIN_SAMPLES), "Test:", len(TEST_SAMPLES), "Eval:", len(EVAL_SAMPLES))
print("Split composition (superpop × sex):")
for name, s in [("train", TRAIN_SAMPLES), ("test", TEST_SAMPLES), ("eval", EVAL_SAMPLES)]:
    sub = hprc_clean[hprc_clean["sample_id"].isin(s)]
    print(name, sub.groupby(["superpopulation", "sex"]).size().unstack(fill_value=0))

## 3. Helper functions (normalize, stream_vcf, classify, truth, metrics)

In [ ]:
def _safe_unlink(path):
    if path and os.path.exists(path):
        try:
            os.unlink(path)
        except Exception:
            pass

def _ensure_filter_in_header(header_text, filter_id="MONOALLELIC", description="Monoallelic site"):
    if f"ID={filter_id}," in header_text or f"ID={filter_id}>" in header_text:
        return header_text
    line = f'##FILTER=<ID={filter_id},Description="{description}">\n'
    if "#CHROM" in header_text:
        parts = header_text.rsplit("#CHROM", 1)
        return parts[0] + line + "#CHROM" + parts[1]
    return header_text + line

def normalize_vcf(in_path, out_path, ref_fa):
    header_out = subprocess.run(
        ["bcftools", "view", "-h", in_path], capture_output=True, text=True, check=True
    )
    fixed_header = _ensure_filter_in_header(header_out.stdout)
    to_norm = in_path
    rehead_plain = None
    if fixed_header != header_out.stdout:
        rehead_plain = out_path.rstrip(".gz").replace(".norm.g.vcf", ".rehead.vcf")
        subprocess.run(
            ["bcftools", "reheader", "-h", "-", "-o", rehead_plain, in_path],
            input=fixed_header, text=True, check=True,
        )
        to_norm = rehead_plain
    norm_unsorted = out_path.replace(".gz", ".unsorted.gz")
    subprocess.run(["bcftools", "norm", "-f", ref_fa, "-m", "-", "-Oz", "-o", norm_unsorted, to_norm], check=True)
    subprocess.run(["bcftools", "sort", "-Oz", "-o", out_path, norm_unsorted], check=True)
    subprocess.run(["tabix", "-p", "vcf", out_path], check=True)
    _safe_unlink(rehead_plain)
    _safe_unlink(norm_unsorted)

def get_info_scalar(variant, key):
    val = variant.info.get(key, (np.nan,))
    return val[0] if isinstance(val, tuple) else float(val)

def classify_variant(ref, alt):
    if len(ref) == 1 and len(alt) == 1:
        return "SNV"
    return "insertion" if len(alt) > len(ref) else "deletion"

In [ ]:
def stream_vcf(vcf_path, ancestry_df, region=None):
    """Stream VCF; return (site_df, geno_df). geno_df has split in train/test/eval for LABELED_SAMPLES."""
    ancestry_map = dict(zip(ancestry_df["sample_id"], ancestry_df["superpopulation"]))
    vcf = pysam.VariantFile(vcf_path)
    if region:
        chrom, start, end = region
        fetch_iter = vcf.fetch(chrom, start, end)
    else:
        fetch_iter = vcf.fetch()
    all_samples = list(vcf.header.samples)
    sample_superpop = {s: ancestry_map.get(s) for s in all_samples}
    site_records, geno_records = [], []
    for variant in fetch_iter:
        alts = variant.alts
        if alts is None or all(a in ("<NON_REF>", "<*>") for a in alts):
            continue
        chrom, pos, ref = variant.chrom, variant.pos, variant.ref
        info_af, info_aq = get_info_scalar(variant, "AF"), get_info_scalar(variant, "AQ")
        is_snv = len(ref) == 1 and all(len(a) == 1 for a in alts if a not in ("<NON_REF>", "<*>"))
        sp_counts = {sp: [0, 0, 0, 0] for sp in SUPERPOPS}
        n_called, n_total = 0, 0
        for sample in all_samples:
            s = variant.samples[sample]
            gt_tuple = s.get("GT", None)
            sp = sample_superpop.get(sample)
            n_total += 1
            if gt_tuple is None or None in gt_tuple:
                if sp in sp_counts:
                    sp_counts[sp][3] += 1
                continue
            a1, a2 = gt_tuple
            n_called += 1
            if sp in sp_counts:
                if a1 == 0 and a2 == 0:
                    sp_counts[sp][0] += 1
                elif a1 != a2:
                    sp_counts[sp][1] += 1
                else:
                    sp_counts[sp][2] += 1
            if sample not in LABELED_SAMPLES or (a1 == 0 and a2 == 0):
                continue
            for alt_idx in {i for i in (a1, a2) if i is not None and i > 0}:
                if alt_idx > len(alts):
                    continue
                alt = alts[alt_idx - 1]
                if alt in ("<NON_REF>", "<*>"):
                    continue
                dp = s.get("DP", np.nan)
                gq = s.get("GQ", np.nan)
                ad, pl, rnc = s.get("AD", None), s.get("PL", None), s.get("RNC", None)
                if ad is not None:
                    ad_clean = [x if x is not None else 0 for x in ad]
                    total_ad = sum(ad_clean)
                    if total_ad > 0 and alt_idx < len(ad_clean):
                        ab, alt_ad, ref_ad = ad_clean[alt_idx] / total_ad, ad_clean[alt_idx], ad_clean[0]
                    else:
                        ab = alt_ad = ref_ad = np.nan
                else:
                    ab = alt_ad = ref_ad = np.nan
                pl_clean = [x if x is not None else np.nan for x in pl] if pl else [np.nan] * 3
                rnc_bad = False
                if rnc is not None:
                    rnc_str = "".join(str(r) for r in rnc) if isinstance(rnc, tuple) else str(rnc)
                    rnc_bad = any(c in rnc_str for c in ("I", "U"))
                split = "train" if sample in TRAIN_SAMPLES else "test" if sample in TEST_SAMPLES else "eval"
                geno_records.append({
                    "chrom": chrom, "pos": pos, "ref": ref, "alt": alt, "sample": sample, "split": split,
                    "superpop": sp, "DP": dp, "GQ": gq, "AB": ab, "alt_AD": alt_ad, "ref_AD": ref_ad,
                    "PL_ref": pl_clean[0], "PL_het": pl_clean[1], "PL_hom": pl_clean[2], "RNC_bad": rnc_bad,
                    "is_snv": is_snv, "is_het": (a1 != a2), "is_hom_alt": (a1 == a2 and a1 > 0),
                    "site_AF": info_af, "site_AQ": info_aq,
                })
        site_records.append({
            "chrom": chrom, "pos": pos, "ref": ref, "alts": ",".join(alts), "is_snv": is_snv,
            "n_called": n_called, "n_total": n_total, "call_rate": n_called / n_total if n_total else np.nan,
            **{f"n_hom_ref_{sp}": sp_counts[sp][0] for sp in SUPERPOPS},
            **{f"n_het_{sp}": sp_counts[sp][1] for sp in SUPERPOPS},
            **{f"n_hom_alt_{sp}": sp_counts[sp][2] for sp in SUPERPOPS},
            **{f"n_miss_{sp}": sp_counts[sp][3] for sp in SUPERPOPS},
        })
    vcf.close()
    return pd.DataFrame(site_records), pd.DataFrame(geno_records)

In [ ]:
def _pick_truth_contig_name(vcf, target_chrom):
    contigs = set(vcf.header.contigs.keys())
    if target_chrom in contigs:
        return target_chrom
    no_chr = target_chrom.replace("chr", "")
    with_chr = "chr" + target_chrom if not target_chrom.startswith("chr") else target_chrom
    if no_chr in contigs:
        return no_chr
    if with_chr in contigs:
        return with_chr
    return target_chrom

def ensure_local_truth_vcfs(samples, truth_vcf_map, cache_dir, ref_fa):
    os.makedirs(cache_dir, exist_ok=True)
    local_paths = {}
    for sample in samples:
        gcs = truth_vcf_map.get(sample)
        if not gcs:
            continue
        fn = os.path.basename(gcs)
        raw_path = os.path.join(cache_dir, fn)
        norm_path = raw_path.replace(".vcf.gz", ".norm.vcf.gz").replace(".vcf.bgz", ".norm.vcf.gz")
        if not os.path.exists(raw_path):
            subprocess.run(["gsutil", "cp", gcs, raw_path], check=True)
        if not os.path.exists(norm_path):
            normalize_vcf(raw_path, norm_path, ref_fa)
        if not os.path.exists(norm_path + ".tbi"):
            subprocess.run(["tabix", "-p", "vcf", norm_path], check=True)
        local_paths[sample] = norm_path
    return local_paths

def build_truth_table(truth_local_vcfs, eval_regions):
    records = []
    for sample, vcf_path in truth_local_vcfs.items():
        vcf = pysam.VariantFile(vcf_path)
        truth_sample = next(iter(vcf.header.samples))
        for target_chrom, (start, end) in eval_regions.items():
            tchrom = _pick_truth_contig_name(vcf, target_chrom)
            try:
                it = vcf.fetch(tchrom, start, end)
            except ValueError:
                continue
            for rec in it:
                ref, alts = rec.ref, rec.alts or ()
                gt = rec.samples[truth_sample].get("GT", None)
                for alt_idx, alt in enumerate(alts, start=1):
                    if alt in ("<NON_REF>", "<*>", None):
                        continue
                    if gt is None or None in gt:
                        truth_state = "missing"
                    else:
                        has1, has2 = (gt[0] == alt_idx), (gt[1] == alt_idx)
                        truth_state = "hom_alt" if (has1 and has2) else ("het" if (has1 or has2) else "ref")
                    records.append({
                        "sample": sample, "chrom": target_chrom, "pos": rec.pos, "ref": ref, "alt": alt,
                        "truth_state": truth_state, "variant_class": classify_variant(ref, alt),
                    })
        vcf.close()
    return pd.DataFrame(records)

def attach_truth_to_genotypes(geno_df, truth_df):
    key_cols = ["sample", "chrom", "pos", "ref", "alt"]
    truth_small = truth_df[key_cols + ["truth_state"]].drop_duplicates()
    geno_base = geno_df.drop(columns=["truth_state", "call_state", "matches_truth"], errors="ignore")
    merged = geno_base.merge(truth_small, how="left", on=key_cols)
    merged["truth_state"] = merged["truth_state"].fillna("unknown")
    merged["call_state"] = merged.apply(
        lambda r: "hom_alt" if r["is_hom_alt"] else ("het" if r["is_het"] else "other"), axis=1
    )
    merged["matches_truth"] = (
        merged["truth_state"].isin(["het", "hom_alt"]) & (merged["truth_state"] == merged["call_state"])
    )
    return merged

def compute_sensitivity_specificity(geno_with_truth, truth_df, variant_class=None, split="test"):
    mask = geno_with_truth["split"] == split
    if variant_class:
        mask &= geno_with_truth["variant_class"] == variant_class
    gsub = geno_with_truth[mask].copy()
    tsub = truth_df.copy()
    if variant_class:
        tsub = tsub[tsub["variant_class"] == variant_class]
    split_samples = TRAIN_SAMPLES if split == "train" else (TEST_SAMPLES if split == "test" else EVAL_SAMPLES)
    tsub = tsub[tsub["sample"].isin(split_samples)]
    TP = int((gsub["matches_truth"] & gsub["truth_state"].isin(["het", "hom_alt"])).sum())
    FP = int((~gsub["truth_state"].isin(["het", "hom_alt"])).sum())
    truth_var = tsub[tsub["truth_state"].isin(["het", "hom_alt"])]
    truth_keys = set(zip(truth_var["sample"], truth_var["chrom"], truth_var["pos"], truth_var["ref"], truth_var["alt"]))
    call_keys = set(zip(gsub["sample"], gsub["chrom"], gsub["pos"], gsub["ref"], gsub["alt"]))
    FN = len(truth_keys - call_keys)
    sens = TP / (TP + FN) if (TP + FN) > 0 else float("nan")
    prec = TP / (TP + FP) if (TP + FP) > 0 else float("nan")
    return {"variant_class": variant_class or "ALL", "split": split, "TP": TP, "FP": FP, "FN": FN,
            "sensitivity": sens, "precision": prec, "n_calls": len(gsub)}

def apply_simple_threshold(geno, gq_threshold=0):
    g = geno.copy()
    # Simple baseline: only filter by genotype quality (GQ)
    keep = (g["GQ"] >= gq_threshold)
    g["passes_threshold"] = keep
    return g

FEATURE_COLS = ["DP", "GQ", "AB", "alt_AD", "ref_AD", "PL_ref", "PL_het", "PL_hom",
                "RNC_bad", "site_AF", "site_AQ", "is_snv", "is_het", "is_hom_alt"]

def make_feature_matrix(df):
    X = df[FEATURE_COLS].copy()
    for col in ["RNC_bad", "is_snv", "is_het", "is_hom_alt"]:
        X[col] = X[col].astype(int)
    return X.fillna(-1)

## 4. Load gVCFs: extract loci → normalize subset → stream → geno_df

Extract only EVAL_REGIONS from each gVCF (saves time), then normalize the subset. Stream to build genotype table and add variant_class.

In [ ]:
# Reference (download if missing)
if not os.path.exists(REF_FASTA):
    subprocess.run(["gsutil", "cp", "gs://prod-drc-broad/longreads/references/GRCh38_noalt/" + REF_FASTA, "."], check=True)
    subprocess.run(["gsutil", "cp", "gs://prod-drc-broad/longreads/references/GRCh38_noalt/" + REF_FASTA + ".fai", "."], check=True)
ref_fa = REF_FASTA

# Download gVCFs, extract EVAL_REGIONS only, normalize subset, then stream
ancestry_df = hprc_clean[["sample_id", "superpopulation", "sex"]].copy()
all_geno = []
for gcs_path in GVCF_GCS_PATHS:
    fn = os.path.basename(gcs_path)
    local_raw = os.path.join(LOCAL_DIR, fn)
    if gcs_path.startswith("gs://") and not os.path.exists(local_raw):
        subprocess.run(["gsutil", "cp", gcs_path, local_raw], check=True)
    if gcs_path.startswith("gs://"):
        vcf_path = local_raw
    else:
        vcf_path = gcs_path
    if not os.path.exists(vcf_path + ".tbi"):
        subprocess.run(["tabix", "-p", "vcf", vcf_path], check=True)
    chrom = fn.replace("subset_", "").replace(".g.vcf.bgz", "").replace(".norm.g.vcf.gz", "")
    start, end = EVAL_REGIONS[chrom]
    subset_path = os.path.join(LOCAL_DIR, f"subset_{chrom}_{start}_{end}.vcf.gz")
    norm_path = subset_path.replace(".vcf.gz", ".norm.g.vcf.gz")
    if not os.path.exists(norm_path):
        if not os.path.exists(subset_path):
            subprocess.run(["bcftools", "view", "-r", f"{chrom}:{start}-{end}", "-Oz", "-o", subset_path, vcf_path], check=True)
            subprocess.run(["tabix", "-p", "vcf", subset_path], check=True)
        normalize_vcf(subset_path, norm_path, ref_fa)
    print("Streaming", norm_path)
    site_df, geno_df = stream_vcf(norm_path, ancestry_df, region=(chrom, start, end))
    all_geno.append(geno_df)
geno_df = pd.concat(all_geno, ignore_index=True)
geno_df["variant_class"] = geno_df.apply(lambda r: classify_variant(r["ref"], r["alt"]), axis=1)
print("Total genotypes:", len(geno_df))
print(geno_df["split"].value_counts())

## 5. Truth VCFs and join

Download/normalize dipcall truth VCFs, build truth table, attach to geno_df.

In [ ]:
TRUTH_CACHE = os.path.join(LOCAL_DIR, "dipcall_truth_vcfs")
truth_samples = [s for s in sorted(LABELED_SAMPLES) if s in truth_vcfs]
print("Truth VCFs for", len(truth_samples), "samples")
truth_local_vcfs = ensure_local_truth_vcfs(truth_samples, truth_vcfs, TRUTH_CACHE, ref_fa)
truth_df = build_truth_table(truth_local_vcfs, EVAL_REGIONS)
geno_with_truth = attach_truth_to_genotypes(geno_df, truth_df)
print("geno_with_truth shape:", geno_with_truth.shape)
print("truth_state:", geno_with_truth["truth_state"].value_counts().head())

## 6. Evaluation: baseline, simple threshold, XGBoost

Baseline and simple-threshold metrics on train and test; XGBoost trained on TRAIN_SAMPLES, threshold chosen on TEST_SAMPLES. EVAL_SAMPLES are not used for training or threshold selection (reserved for future Hiphase).

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score

# Baseline (unfiltered)
metrics_baseline = []
for split in ["train", "test"]:
    metrics_baseline.append(compute_sensitivity_specificity(geno_with_truth, truth_df, variant_class=None, split=split))
    for vc in ["SNV", "insertion", "deletion"]:
        metrics_baseline.append(compute_sensitivity_specificity(geno_with_truth, truth_df, variant_class=vc, split=split))
print("Baseline (unfiltered):")
for m in metrics_baseline:
    print(m)

# Simple threshold
geno_thr = apply_simple_threshold(geno_with_truth)
metrics_threshold = []
for split in ["train", "test"]:
    sub = geno_thr[geno_thr["passes_threshold"]]
    metrics_threshold.append(compute_sensitivity_specificity(sub, truth_df, variant_class=None, split=split))
    for vc in ["SNV", "insertion", "deletion"]:
        metrics_threshold.append(compute_sensitivity_specificity(sub, truth_df, variant_class=vc, split=split))
print("\nSimple-threshold filtered:")
for m in metrics_threshold:
    print(m)

In [ ]:
# XGBoost: separate model per variant class (SNV, insertion, deletion); train on TRAIN_SAMPLES, threshold on TEST
import time
_t0 = time.perf_counter()
geno_labeled = geno_with_truth[geno_with_truth["truth_state"].isin(["het", "hom_alt", "ref"])].copy()
geno_labeled["label"] = geno_labeled["matches_truth"].astype(int)

xgb_models = {}
VARIANT_CLASSES = ["SNV", "insertion", "deletion"]

for vclass in VARIANT_CLASSES:
    train_mask = (geno_labeled["split"] == "train") & (geno_labeled["variant_class"] == vclass)
    train_df = geno_labeled[train_mask].copy()
    if train_df.empty:
        print(f"No training data for {vclass}, skipping")
        continue
    # Subsample per stratum (is_het x is_hom_alt x matches_truth)
    def _subsample(g):
        n = min(len(g), SUBSAMPLE_PER_STRATUM)
        return g.sample(n=n, random_state=RANDOM_SEED) if len(g) > n else g
    train_df = train_df.groupby(["is_het", "is_hom_alt", "matches_truth"], group_keys=False).apply(_subsample).reset_index(drop=True)
    X_train = make_feature_matrix(train_df)
    y_train = train_df["label"].values
    model = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=RANDOM_SEED, use_label_encoder=False, eval_metric="logloss")
    model.fit(X_train, y_train)
    xgb_models[vclass] = model
    print(f"  {vclass} trained in {time.perf_counter() - _t0:.1f}s")
    test_mask = (geno_labeled["split"] == "test") & (geno_labeled["variant_class"] == vclass)
    test_df_vc = geno_labeled[test_mask]
    if not test_df_vc.empty:
        X_test = make_feature_matrix(test_df_vc)
        y_test = test_df_vc["label"].values
        auc_v = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        ap_v = average_precision_score(y_test, model.predict_proba(X_test)[:, 1])
        print(f"{vclass} XGBoost: test ROC-AUC={auc_v:.3f}, PR-AUC={ap_v:.3f} (n={len(test_df_vc):,})")

# Score all genotypes with the appropriate model per variant class
_t1 = time.perf_counter()
geno_xgb = geno_with_truth.copy()
geno_xgb["xgb_score"] = np.nan
for vclass, model in xgb_models.items():
    mask = geno_xgb["variant_class"] == vclass
    if not mask.any():
        continue
    geno_xgb.loc[mask, "xgb_score"] = model.predict_proba(make_feature_matrix(geno_xgb[mask]))[:, 1]
print(f"Scoring all genotypes: {time.perf_counter() - _t1:.1f}s")

# Per-class threshold: choose operating point (sensitivity and max 1-specificity) on test.
# Specify per variant class with dicts, or a single number to use for all classes.
# Re-run this cell and the plot cell after changing.
MIN_SENSITIVITY = {"SNV": 0.8, "insertion": 0.5, "deletion": 0.8}
MAX_ONE_MINUS_SPECIFICITY = {"SNV": 0.1, "insertion": 0.15, "deletion": 0.1}  # FPR <= this value

# Fixed threshold: set to a number (e.g. 0.96) to use that threshold for all variant classes; set to None to use min_sens/max_fpr search.
FIXED_XGB_THRESHOLD = 0.96

def _target(per_class, vclass, default):
    if isinstance(per_class, dict):
        return per_class.get(vclass, default)
    return per_class

# Pre-extract test (y_true, y_score) per variant class so we can sweep thresholds without touching full dataframe
threshold_grid = np.linspace(0.05, 0.995, 95)
THRESH_XGB = {}
THRESH_XGB_FALLBACK = {}  # True if no threshold met both min_sens and max_fpr (we used best sens with fpr<=max_fpr)
_t2_start = time.perf_counter()
if FIXED_XGB_THRESHOLD is not None:
    for vclass in VARIANT_CLASSES:
        THRESH_XGB[vclass] = FIXED_XGB_THRESHOLD if vclass in xgb_models else 0.5
        THRESH_XGB_FALLBACK[vclass] = False
    print(f"Using fixed XGBoost threshold for all classes: {FIXED_XGB_THRESHOLD}")
else:
    for vclass in VARIANT_CLASSES:
        _tv = time.perf_counter()
        if vclass not in xgb_models:
            THRESH_XGB[vclass] = 0.5
            THRESH_XGB_FALLBACK[vclass] = False
            continue
        test_mask = (geno_xgb["split"] == "test") & (geno_xgb["variant_class"] == vclass)
        min_sens = _target(MIN_SENSITIVITY, vclass, 0.8)
        max_fpr = _target(MAX_ONE_MINUS_SPECIFICITY, vclass, 0.1)
        test_df_vc = geno_xgb.loc[test_mask, ["matches_truth", "xgb_score"]].dropna(subset=["xgb_score"])
        y_true = test_df_vc["matches_truth"].astype(int).values
        y_score = test_df_vc["xgb_score"].values
        best_t = 0.5
        best_sens = -1.0
        fallback_t = 0.5
        fallback_sens = -1.0
        for t in threshold_grid:
            y_pred = (y_score >= t).astype(int)
            tp = ((y_true == 1) & (y_pred == 1)).sum()
            fn = ((y_true == 1) & (y_pred == 0)).sum()
            fp = ((y_true == 0) & (y_pred == 1)).sum()
            tn = ((y_true == 0) & (y_pred == 0)).sum()
            sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            if sens >= min_sens and fpr <= max_fpr and sens > best_sens:
                best_sens = sens
                best_t = t
            if fpr <= max_fpr and sens > fallback_sens:
                fallback_sens = sens
                fallback_t = t
        used_fallback = best_sens < 0
        THRESH_XGB_FALLBACK[vclass] = used_fallback
        THRESH_XGB[vclass] = best_t if best_sens >= 0 else fallback_t
        if used_fallback and fallback_sens >= 0:
            print(f"  {vclass}: no threshold with sens>={min_sens} AND FPR<={max_fpr}; using fallback (best sens at FPR<={max_fpr}): sens={fallback_sens:.3f}, th={fallback_t:.3f}")
        print(f"  {vclass}: {time.perf_counter() - _tv:.1f}s (chosen th={THRESH_XGB[vclass]:.3f})")

# Set xgb_pass vectorized (no apply over full dataframe)
geno_xgb["xgb_pass"] = False
for vc, th in THRESH_XGB.items():
    mask = (geno_xgb["variant_class"] == vc) & np.isfinite(geno_xgb["xgb_score"]) & (geno_xgb["xgb_score"] >= th)
    geno_xgb.loc[mask, "xgb_pass"] = True
print(f"Threshold search total: {time.perf_counter() - _t2_start:.1f}s")

# Final metrics (train + test)
_t3 = time.perf_counter()

print("XGBoost thresholds (per-class targets on test):", THRESH_XGB)
if FIXED_XGB_THRESHOLD is not None:
    print("  Using fixed threshold:", FIXED_XGB_THRESHOLD)
else:
    if isinstance(MIN_SENSITIVITY, dict):
        print("  Targets: min_sensitivity =", MIN_SENSITIVITY, ", max_1_specificity =", MAX_ONE_MINUS_SPECIFICITY)
    else:
        print("  Targets: min_sensitivity = %.2f, max_1_specificity = %.2f" % (MIN_SENSITIVITY, MAX_ONE_MINUS_SPECIFICITY))
    met = [vc for vc in VARIANT_CLASSES if not THRESH_XGB_FALLBACK.get(vc, False)]
    fallback = [vc for vc in VARIANT_CLASSES if THRESH_XGB_FALLBACK.get(vc, False)]
    if met:
        print("  Met targets:", met)
    if fallback:
        print("  Fallback (no th met both min_sens and max_fpr):", fallback, "- consider relaxing min_sensitivity or max_1_specificity for these.")
metrics_xgb = []
for split in ["train", "test"]:
    sub = geno_xgb[geno_xgb["xgb_pass"]]
    metrics_xgb.append(compute_sensitivity_specificity(sub, truth_df, variant_class=None, split=split))
    for vc in VARIANT_CLASSES:
        metrics_xgb.append(compute_sensitivity_specificity(sub, truth_df, variant_class=vc, split=split))
print("Final metrics: {:.1f}s".format(time.perf_counter() - _t3))
print("XGBoost cell total: {:.1f}s".format(time.perf_counter() - _t0))
print("\nXGBoost filtered (per-class thresholds):")
for m in metrics_xgb:
    print(m)

## 7. Plots: GQ ROC, XGBoost ROC/PR, summary table

ROC for simple GQ thresholding; ROC and PR for XGBoost; summary metrics table.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# Prep: test data for simple threshold and XGBoost (known truth only)
test_g = geno_thr[(geno_thr["split"] == "test") & (geno_thr["truth_state"].isin(["het", "hom_alt", "ref"]))].copy()
test_g["y_true"] = ((test_g["truth_state"].isin(["het", "hom_alt"])) & (test_g["matches_truth"])).astype(int)
test_xgb = geno_xgb[(geno_xgb["split"] == "test") & (geno_xgb["truth_state"].isin(["het", "hom_alt", "ref"]))]
gq_values = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 99]

# Sensitivity vs 1-Specificity: same plot type, both methods per panel (SNV, insertion, deletion)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, vc in zip(axes, ["SNV", "insertion", "deletion"]):
    sub_g = test_g[test_g["variant_class"] == vc]
    sub_x = test_xgb[test_xgb["variant_class"] == vc]
    sub_x = sub_x[np.isfinite(sub_x["xgb_score"])]
    if not sub_g.empty:
        y_true_all = sub_g["y_true"].values
        tpr_list, fpr_list = [], []
        for gq in gq_values:
            pass_ = (sub_g["GQ"] >= gq)
            y_pred = pass_.astype(int)
            tn = ((y_true_all == 0) & (y_pred == 0)).sum()
            fp = ((y_true_all == 0) & (y_pred == 1)).sum()
            fn = ((y_true_all == 1) & (y_pred == 0)).sum()
            tp = ((y_true_all == 1) & (y_pred == 1)).sum()
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            tpr_list.append(tpr)
            fpr_list.append(fpr)
        ax.plot(fpr_list, tpr_list, "o-", label="Simple threshold (GQ)", color="C0")
        for i, gq in enumerate(gq_values):
            ax.annotate(str(gq), (fpr_list[i], tpr_list[i]), fontsize=7)
    if not sub_x.empty:
        y_true = sub_x["matches_truth"].astype(int).values
        y_score = sub_x["xgb_score"].values
        fpr, tpr, _ = roc_curve(y_true, y_score)
        ax.plot(fpr, tpr, "-", label="XGBoost (AUC=%.3f)" % auc(fpr, tpr), color="C1")
        # Multiple XGBoost score cutoffs to show sensitivity/specificity tradeoff
        xgb_thresh_candidates = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 0.985, 0.99, 0.992, 0.995, 0.998]
        chosen_th = THRESH_XGB.get(vc)
        thresh_to_plot = list(xgb_thresh_candidates)
        if chosen_th is not None and chosen_th not in thresh_to_plot:
            thresh_to_plot = sorted(thresh_to_plot + [chosen_th])
        for th in thresh_to_plot:
            y_pred = (y_score >= th).astype(int)
            tp = ((y_true == 1) & (y_pred == 1)).sum()
            fn = ((y_true == 1) & (y_pred == 0)).sum()
            fp = ((y_true == 0) & (y_pred == 1)).sum()
            tn = ((y_true == 0) & (y_pred == 0)).sum()
            tpr_at = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            fpr_at = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            ax.plot(fpr_at, tpr_at, "o", color="C1", markersize=6, markeredgecolor="k", markeredgewidth=0.5, zorder=4)
            ax.annotate(f"{th:.2f}", (fpr_at, tpr_at), xytext=(3, 3), textcoords="offset points", fontsize=6, color="C1")
        # Single chosen point: plot one star at the actual chosen threshold's operating point
        if chosen_th is not None and not sub_x.empty:
            y_pred = (y_score >= chosen_th).astype(int)
            tp = ((y_true == 1) & (y_pred == 1)).sum()
            fn = ((y_true == 1) & (y_pred == 0)).sum()
            fp = ((y_true == 0) & (y_pred == 1)).sum()
            tn = ((y_true == 0) & (y_pred == 0)).sum()
            tpr_at = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            fpr_at = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            ax.plot(fpr_at, tpr_at, "*", color="C1", markersize=16, markeredgecolor="k", markeredgewidth=1, zorder=5)
            fallback = THRESH_XGB_FALLBACK.get(vc, False)
            label = f"th={chosen_th:.3f}\nsens={tpr_at:.2f}, FPR={fpr_at:.2f}" + ("\n(fallback)" if fallback else " (chosen)")
            ax.annotate(label, (fpr_at, tpr_at), xytext=(5, 5), textcoords="offset points", fontsize=7, color="C1", fontweight="bold" if fallback else "normal")
    ax.set_xlabel("1 - Specificity"); ax.set_ylabel("Sensitivity"); ax.set_title(f"{vc} (test)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim(0.0, 1.0); ax.set_ylim(0.0, 1.0)
plt.suptitle("Sensitivity vs 1-Specificity: simple threshold vs XGBoost", y=1.02)
plt.tight_layout()
plt.show()

# Precision vs Recall: same plot type, both methods per panel (SNV, insertion, deletion)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, vc in zip(axes, ["SNV", "insertion", "deletion"]):
    sub_g = test_g[test_g["variant_class"] == vc]
    sub_x = test_xgb[test_xgb["variant_class"] == vc]
    sub_x = sub_x[np.isfinite(sub_x["xgb_score"])]
    if not sub_g.empty:
        y_true_all = sub_g["y_true"].values
        rec_list, prec_list = [], []
        for gq in gq_values:
            pass_ = (sub_g["GQ"] >= gq)
            y_pred = pass_.astype(int)
            fp = ((y_true_all == 0) & (y_pred == 1)).sum()
            fn = ((y_true_all == 1) & (y_pred == 0)).sum()
            tp = ((y_true_all == 1) & (y_pred == 1)).sum()
            rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rec_list.append(rec)
            prec_list.append(prec)
        ax.plot(rec_list, prec_list, "o-", label="Simple threshold (GQ)", color="C0")
        for i, gq in enumerate(gq_values):
            ax.annotate(str(gq), (rec_list[i], prec_list[i]), fontsize=7)
    if not sub_x.empty:
        y_true = sub_x["matches_truth"].astype(int).values
        y_score = sub_x["xgb_score"].values
        prec, rec, _ = precision_recall_curve(y_true, y_score)
        ax.plot(rec, prec, "-", label="XGBoost (AP=%.3f)" % average_precision_score(y_true, y_score), color="C1")
        # Multiple XGBoost score cutoffs for tradeoff comparison
        xgb_thresh_candidates = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 0.985, 0.99, 0.992, 0.995, 0.998]
        chosen_th = THRESH_XGB.get(vc)
        thresh_to_plot = list(xgb_thresh_candidates)
        if chosen_th is not None and chosen_th not in thresh_to_plot:
            thresh_to_plot = sorted(thresh_to_plot + [chosen_th])
        for th in thresh_to_plot:
            y_pred = (y_score >= th).astype(int)
            tp = ((y_true == 1) & (y_pred == 1)).sum()
            fn = ((y_true == 1) & (y_pred == 0)).sum()
            fp = ((y_true == 0) & (y_pred == 1)).sum()
            rec_at = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            prec_at = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            ax.plot(rec_at, prec_at, "o", color="C1", markersize=6, markeredgecolor="k", markeredgewidth=0.5, zorder=4)
            ax.annotate(f"{th:.2f}", (rec_at, prec_at), xytext=(3, 3), textcoords="offset points", fontsize=6, color="C1")
        # Single chosen point at the actual chosen threshold's operating point
        if chosen_th is not None and not sub_x.empty:
            y_pred = (y_score >= chosen_th).astype(int)
            tp = ((y_true == 1) & (y_pred == 1)).sum()
            fn = ((y_true == 1) & (y_pred == 0)).sum()
            fp = ((y_true == 0) & (y_pred == 1)).sum()
            rec_at = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            prec_at = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            ax.plot(rec_at, prec_at, "*", color="C1", markersize=16, markeredgecolor="k", markeredgewidth=1, zorder=5)
            fallback = THRESH_XGB_FALLBACK.get(vc, False)
            label = f"th={chosen_th:.3f}\nrec={rec_at:.2f}, prec={prec_at:.2f}" + ("\n(fallback)" if fallback else " (chosen)")
            ax.annotate(label, (rec_at, prec_at), xytext=(5, 5), textcoords="offset points", fontsize=7, color="C1", fontweight="bold" if fallback else "normal")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title(f"{vc} (test)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim(0.0, 1.0); ax.set_ylim(0.0, 1.0)
plt.suptitle("Precision vs Recall: simple threshold vs XGBoost", y=1.02)
plt.tight_layout()
plt.show()

### Interactive plots (hover to see operating points)

Run the cell below to get interactive Sensitivity vs 1-Specificity and Precision vs Recall plots. Hover over any point to see threshold, sensitivity, FPR (or recall/precision) and pick the operating point you want; then set `MIN_SENSITIVITY` / `MAX_ONE_MINUS_SPECIFICITY` or the chosen threshold in the XGBoost cell accordingly.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import roc_curve, precision_recall_curve

# Reuse test_g, test_xgb, gq_values from the static plot cell
xgb_thresh_candidates = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 0.985, 0.99, 0.992, 0.995, 0.998]
VARIANT_CLASSES_PLOT = ["SNV", "insertion", "deletion"]

# --- Sensitivity vs 1-Specificity (interactive) ---
fig_roc = make_subplots(rows=1, cols=3, subplot_titles=[f"{vc} (test)" for vc in VARIANT_CLASSES_PLOT],
    x_title="1 - Specificity", y_title="Sensitivity")
for col, vc in enumerate(VARIANT_CLASSES_PLOT, start=1):
    sub_g = test_g[test_g["variant_class"] == vc]
    sub_x = test_xgb[test_xgb["variant_class"] == vc]
    sub_x = sub_x[np.isfinite(sub_x["xgb_score"])]
    if not sub_g.empty:
        y_true_all = sub_g["y_true"].values
        fpr_list, tpr_list, hover_gq = [], [], []
        for gq in gq_values:
            pass_ = (sub_g["GQ"] >= gq)
            y_pred = pass_.astype(int)
            tn = ((y_true_all == 0) & (y_pred == 0)).sum()
            fp = ((y_true_all == 0) & (y_pred == 1)).sum()
            fn = ((y_true_all == 1) & (y_pred == 0)).sum()
            tp = ((y_true_all == 1) & (y_pred == 1)).sum()
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            fpr_list.append(fpr); tpr_list.append(tpr); hover_gq.append(f"GQ≥{gq}<br>Sensitivity={tpr:.3f}<br>1-Specificity={fpr:.3f}")
        fig_roc.add_trace(go.Scatter(x=fpr_list, y=tpr_list, mode="lines+markers", name="Simple (GQ)", line=dict(color="rgb(31,119,180)"),
            text=hover_gq, hovertemplate="%{text}<extra></extra>", legendgroup="simple"), row=1, col=col)
    if not sub_x.empty:
        y_true = sub_x["matches_truth"].astype(int).values
        y_score = sub_x["xgb_score"].values
        fpr_curve, tpr_curve, _ = roc_curve(y_true, y_score)
        fig_roc.add_trace(go.Scatter(x=fpr_curve, y=tpr_curve, mode="lines", name="XGBoost (curve)", line=dict(color="rgb(255,127,14)"), legendgroup="xgb"), row=1, col=col)
        chosen_th = THRESH_XGB.get(vc)
        thresh_to_plot = list(xgb_thresh_candidates)
        if chosen_th is not None and chosen_th not in thresh_to_plot:
            thresh_to_plot = sorted(thresh_to_plot + [chosen_th])
        fpr_pts, tpr_pts, th_pts, hover_pts = [], [], [], []
        for th in thresh_to_plot:
            y_pred = (y_score >= th).astype(int)
            tp = ((y_true == 1) & (y_pred == 1)).sum()
            fn = ((y_true == 1) & (y_pred == 0)).sum()
            fp = ((y_true == 0) & (y_pred == 1)).sum()
            tn = ((y_true == 0) & (y_pred == 0)).sum()
            tpr_at = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            fpr_at = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            fpr_pts.append(fpr_at); tpr_pts.append(tpr_at); th_pts.append(th)
            is_chosen = chosen_th is not None and abs(th - chosen_th) < 1e-6
            tag = " (fallback)" if is_chosen and THRESH_XGB_FALLBACK.get(vc, False) else (" (chosen)" if is_chosen else "")
            hover_pts.append(f"XGBoost th={th:.3f}{tag}<br>Sensitivity={tpr_at:.3f}<br>1-Specificity={fpr_at:.3f}")
        fig_roc.add_trace(go.Scatter(x=fpr_pts, y=tpr_pts, mode="markers", name="XGBoost (points)", marker=dict(size=8, color="rgb(255,127,14)"),
            text=hover_pts, hovertemplate="%{text}<extra></extra>", legendgroup="xgb"), row=1, col=col)
    fig_roc.update_xaxes(range=[0, 1], row=1, col=col)
    fig_roc.update_yaxes(range=[0, 1], row=1, col=col)
fig_roc.update_layout(title_text="Sensitivity vs 1-Specificity (hover for operating points)", height=420, showlegend=True)
fig_roc.show()

# --- Precision vs Recall (interactive) ---
fig_pr = make_subplots(rows=1, cols=3, subplot_titles=[f"{vc} (test)" for vc in VARIANT_CLASSES_PLOT],
    x_title="Recall", y_title="Precision")
for col, vc in enumerate(VARIANT_CLASSES_PLOT, start=1):
    sub_g = test_g[test_g["variant_class"] == vc]
    sub_x = test_xgb[test_xgb["variant_class"] == vc]
    sub_x = sub_x[np.isfinite(sub_x["xgb_score"])]
    if not sub_g.empty:
        y_true_all = sub_g["y_true"].values
        rec_list, prec_list, hover_gq = [], [], []
        for gq in gq_values:
            pass_ = (sub_g["GQ"] >= gq)
            y_pred = pass_.astype(int)
            fp = ((y_true_all == 0) & (y_pred == 1)).sum()
            fn = ((y_true_all == 1) & (y_pred == 0)).sum()
            tp = ((y_true_all == 1) & (y_pred == 1)).sum()
            rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rec_list.append(rec); prec_list.append(prec)
            hover_gq.append(f"GQ≥{gq}<br>Recall={rec:.3f}<br>Precision={prec:.3f}")
        fig_pr.add_trace(go.Scatter(x=rec_list, y=prec_list, mode="lines+markers", name="Simple (GQ)", line=dict(color="rgb(31,119,180)"),
            text=hover_gq, hovertemplate="%{text}<extra></extra>", legendgroup="simple"), row=1, col=col)
    if not sub_x.empty:
        y_true = sub_x["matches_truth"].astype(int).values
        y_score = sub_x["xgb_score"].values
        prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_score)
        fig_pr.add_trace(go.Scatter(x=rec_curve, y=prec_curve, mode="lines", name="XGBoost (curve)", line=dict(color="rgb(255,127,14)"), legendgroup="xgb"), row=1, col=col)
        chosen_th = THRESH_XGB.get(vc)
        thresh_to_plot = list(xgb_thresh_candidates)
        if chosen_th is not None and chosen_th not in thresh_to_plot:
            thresh_to_plot = sorted(thresh_to_plot + [chosen_th])
        rec_pts, prec_pts, hover_pts = [], [], []
        for th in thresh_to_plot:
            y_pred = (y_score >= th).astype(int)
            tp = ((y_true == 1) & (y_pred == 1)).sum()
            fn = ((y_true == 1) & (y_pred == 0)).sum()
            fp = ((y_true == 0) & (y_pred == 1)).sum()
            rec_at = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            prec_at = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rec_pts.append(rec_at); prec_pts.append(prec_at)
            is_chosen = chosen_th is not None and abs(th - chosen_th) < 1e-6
            tag = " (fallback)" if is_chosen and THRESH_XGB_FALLBACK.get(vc, False) else (" (chosen)" if is_chosen else "")
            hover_pts.append(f"XGBoost th={th:.3f}{tag}<br>Recall={rec_at:.3f}<br>Precision={prec_at:.3f}")
        fig_pr.add_trace(go.Scatter(x=rec_pts, y=prec_pts, mode="markers", name="XGBoost (points)", marker=dict(size=8, color="rgb(255,127,14)"),
            text=hover_pts, hovertemplate="%{text}<extra></extra>", legendgroup="xgb"), row=1, col=col)
    fig_pr.update_xaxes(range=[0, 1], row=1, col=col)
    fig_pr.update_yaxes(range=[0, 1], row=1, col=col)
fig_pr.update_layout(title_text="Precision vs Recall (hover for operating points)", height=420, showlegend=True)
fig_pr.show()

In [ ]:
# Summary table: baseline vs simple threshold vs XGBoost (test split), separate rows for ALL, SNV, insertion, deletion
def to_rows(metrics_list, method_name):
    return [
        {"method": method_name, "variant_class": m["variant_class"], "sensitivity": m["sensitivity"], "precision": m["precision"], "TP": m["TP"], "FP": m["FP"], "FN": m["FN"]}
        for m in metrics_list if m.get("split") == "test"
    ]
rows = to_rows(metrics_baseline, "baseline") + to_rows(metrics_threshold, "simple_threshold") + to_rows(metrics_xgb, "xgboost")
summary = pd.DataFrame(rows)
# Order: ALL first, then SNV, insertion, deletion; then by method
summary["_ord"] = summary["variant_class"].map({"ALL": 0, "SNV": 1, "insertion": 2, "deletion": 3})
summary = summary.sort_values(["_ord", "method"]).drop(columns=["_ord"])
print(summary.to_string(index=False))

## 8. Phasing study (chr20:40–50 Mbp)

Evaluate how unfiltered vs XGBoost-filtered VCFs affect physical phasing. Extract chr20:40–50 Mbp from the per-chrom gVCF, produce unfiltered and filtered multi-sample VCFs, split by sample, stream only that region from GCS BAMs (no full download), and run HiPhase with 15x and 30x BAMs.

In [ ]:
# Phasing region and output dir; GCS auth for samtools streaming
PHASE_CHROM = "chr20"
PHASE_START = 40_000_000
PHASE_END = 50_000_000
PHASE_REGION = f"{PHASE_CHROM}:{PHASE_START}-{PHASE_END}"
PHASE_DIR = os.path.join(LOCAL_DIR, "phasing_study")
os.makedirs(PHASE_DIR, exist_ok=True)
os.makedirs(os.path.join(PHASE_DIR, "bams"), exist_ok=True)

# GCS_OAUTH_TOKEN so samtools can read from gs:// (htslib)
try:
    token = subprocess.run(
        ["gcloud", "auth", "application-default", "print-access-token"],
        capture_output=True, text=True, check=True
    ).stdout.strip()
    os.environ["GCS_OAUTH_TOKEN"] = token
    print("GCS_OAUTH_TOKEN set for samtools/htslib")
except (subprocess.CalledProcessError, FileNotFoundError) as e:
    print("Warning: could not set GCS_OAUTH_TOKEN:", e)

# Restrict phasing study to EVAL_SAMPLES (held out from training/threshold selection)
PHASE_SAMPLES_SUBSET = EVAL_SAMPLES

In [ ]:
# Extract chr20:40-50 Mbp from gVCF and normalize (unfiltered multi-sample VCF)
gcs_chr20 = GVCF_GCS_PATHS[0]
local_chr20 = os.path.join(LOCAL_DIR, os.path.basename(gcs_chr20))
if gcs_chr20.startswith("gs://") and not os.path.exists(local_chr20):
    subprocess.run(["gsutil", "cp", gcs_chr20, local_chr20], check=True)
if not os.path.exists(local_chr20 + ".tbi"):
    subprocess.run(["tabix", "-p", "vcf", local_chr20], check=True)

phase_subset_path = os.path.join(PHASE_DIR, f"phasing_unfiltered_{PHASE_CHROM}_{PHASE_START//1e6:.0f}M_{PHASE_END//1e6:.0f}M.vcf.gz")
phase_norm_path = phase_subset_path.replace(".vcf.gz", ".norm.vcf.gz")
if not os.path.exists(phase_norm_path):
    if not os.path.exists(phase_subset_path):
        subprocess.run(
            ["bcftools", "view", "-r", PHASE_REGION, "-Oz", "-o", phase_subset_path, local_chr20],
            check=True, env={**os.environ}
        )
        subprocess.run(["tabix", "-p", "vcf", phase_subset_path], check=True)
    normalize_vcf(phase_subset_path, phase_norm_path, ref_fa)
if not os.path.exists(phase_norm_path + ".tbi"):
    subprocess.run(["tabix", "-p", "vcf", phase_norm_path], check=True)
print("Unfiltered phasing VCF:", phase_norm_path)

In [ ]:
# Samples to phase: in VCF, have both 15x and 30x BAMs, and in EVAL_SAMPLES
with pysam.VariantFile(phase_norm_path) as vf:
    vcf_samples = set(vf.header.samples)
bam_samples = set(bams_15x.keys()) & set(bams_30x.keys())
PHASE_SAMPLES = sorted(vcf_samples & bam_samples & PHASE_SAMPLES_SUBSET)
print("PHASE_SAMPLES (EVAL_SAMPLES with BAMs):", len(PHASE_SAMPLES), PHASE_SAMPLES[:5], "..." if len(PHASE_SAMPLES) > 5 else "")

In [ ]:
# Produce filtered multi-sample VCF: stream normalized VCF, set failing GTs to ./. per XGBoost
phase_filtered_path = phase_norm_path.replace(".norm.vcf.gz", ".filtered.norm.vcf.gz")
if not os.path.exists(phase_filtered_path):
    try:
        from tqdm import tqdm
    except ImportError:
        def tqdm(itr, desc="", unit="it", **kwargs):
            return itr
    in_vcf = pysam.VariantFile(phase_norm_path)
    # Build output header without FORMAT "ref" (gVCF field that pysam new_record cannot set)
    header_lines = []
    for line in str(in_vcf.header).splitlines():
        if line.startswith("##FORMAT=") and ("ID=ref," in line or "ID=ref>" in line):
            continue
        header_lines.append(line)
    samples_order = list(in_vcf.header.samples)
    fmt_cols = ["GT", "DP", "GQ", "AD", "PL", "RNC"]
    raw_path = phase_filtered_path.replace(".vcf.gz", ".vcf")

    def fmt_val(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "."
        if isinstance(v, (list, tuple)):
            return ",".join("." if x is None or (isinstance(x, float) and np.isnan(x)) else (str(int(x)) if isinstance(x, (int, float)) and x == int(x) else str(x)) for x in v)
        return str(int(v)) if isinstance(v, (int, float)) and v == int(v) else str(v)

    with open(raw_path, "w") as out:
        out.write("\n".join(header_lines) + "\n")
        n_written = 0
        for rec in tqdm(in_vcf.fetch(PHASE_CHROM, PHASE_START, PHASE_END), desc="Filtering variants", unit="vars"):
            alts = rec.alts
            if alts is None or all(a in ("<NON_REF>", "<*>") for a in alts):
                continue
            ref, chrom, pos = rec.ref, rec.chrom, rec.pos
            info_af = get_info_scalar(rec, "AF")
            info_aq = get_info_scalar(rec, "AQ")
            is_snv = len(ref) == 1 and all(len(a) == 1 for a in alts if a not in ("<NON_REF>", "<*>"))
            # Collect all (sample, vc, feature_dict) for this variant; batch score with XGBoost
            rows_to_score = []
            for sample in rec.samples:
                s = rec.samples[sample]
                gt = s.get("GT")
                if gt is None or None in gt:
                    continue
                a1, a2 = gt
                if a1 == 0 and a2 == 0:
                    continue
                for alt_idx in {i for i in (a1, a2) if i and i <= len(alts)}:
                    alt = alts[alt_idx - 1]
                    if alt in ("<NON_REF>", "<*>"):
                        continue
                    vc = classify_variant(ref, alt)
                    if vc not in xgb_models:
                        continue
                    dp = s.get("DP", np.nan)
                    gq = s.get("GQ", np.nan)
                    ad, pl, rnc = s.get("AD"), s.get("PL"), s.get("RNC")
                    if ad is not None:
                        ad_clean = [x if x is not None else 0 for x in ad]
                        total = sum(ad_clean)
                        ab = ad_clean[alt_idx] / total if total and alt_idx < len(ad_clean) else np.nan
                        alt_ad = ad_clean[alt_idx] if alt_idx < len(ad_clean) else np.nan
                        ref_ad = ad_clean[0] if ad_clean else np.nan
                    else:
                        ab = alt_ad = ref_ad = np.nan
                    pl_clean = [x if x is not None else np.nan for x in pl] if pl else [np.nan] * 3
                    rnc_bad = False
                    if rnc is not None:
                        rnc_str = "".join(str(r) for r in rnc) if isinstance(rnc, tuple) else str(rnc)
                        rnc_bad = any(c in rnc_str for c in ("I", "U"))
                    rows_to_score.append((sample, vc, {"DP": dp, "GQ": gq, "AB": ab, "alt_AD": alt_ad, "ref_AD": ref_ad,
                        "PL_ref": pl_clean[0], "PL_het": pl_clean[1], "PL_hom": pl_clean[2],
                        "RNC_bad": rnc_bad, "site_AF": info_af, "site_AQ": info_aq,
                        "is_snv": is_snv, "is_het": (a1 != a2), "is_hom_alt": (a1 == a2 and a1 > 0)}))
            samples_to_miss = set()
            if rows_to_score:
                batch_df = pd.DataFrame([r[2] for r in rows_to_score])
                X_all = make_feature_matrix(batch_df)
                samples_per_row = [r[0] for r in rows_to_score]
                vc_per_row = [r[1] for r in rows_to_score]
                for vc in xgb_models:
                    idx = [i for i, v in enumerate(vc_per_row) if v == vc]
                    if not idx:
                        continue
                    X_vc = X_all.iloc[idx]
                    scores = xgb_models[vc].predict_proba(make_feature_matrix(X_vc))[:, 1]
                    th = THRESH_XGB.get(vc, 0.5)
                    for j, score in enumerate(scores):
                        if score < th:
                            samples_to_miss.add(samples_per_row[idx[j]])

            def sample_fmt(samp):
                if samp in samples_to_miss:
                    return ":".join(["." for _ in fmt_cols])
                s = rec.samples[samp]
                parts = []
                for k in fmt_cols:
                    v = s.get(k)
                    if k == "GT" and v is not None and None not in v:
                        parts.append("/".join(str(a) for a in v))
                    else:
                        parts.append(fmt_val(v))
                return ":".join(parts)
            id_val = rec.id if rec.id else "."
            qual_val = rec.qual if rec.qual is not None else "."
            filter_val = ";".join(rec.filter) if rec.filter else "."
            info_parts = []
            for key, val in rec.info.items():
                if val is None:
                    continue
                if isinstance(val, (list, tuple)):
                    info_parts.append(f"{key}={','.join(str(x) for x in val)}")
                else:
                    info_parts.append(f"{key}={val}")
            info_str = ";".join(info_parts) if info_parts else "."
            fmt_str = ":".join(fmt_cols)
            sample_strs = [sample_fmt(s) for s in samples_order]
            out.write("\t".join([str(chrom), str(pos), id_val, ref, ",".join(rec.alts), str(qual_val), filter_val, info_str, fmt_str] + sample_strs) + "\n")
            n_written += 1
    in_vcf.close()
    with open(raw_path, "rb") as f_in:
        with open(phase_filtered_path, "wb") as f_out:
            subprocess.run(["bgzip", "-c"], stdin=f_in, stdout=f_out, check=True)
    os.unlink(raw_path)
    subprocess.run(["tabix", "-p", "vcf", phase_filtered_path], check=True)
    print("Filtered phasing VCF:", phase_filtered_path, "variants:", n_written)
else:
    print("Filtered phasing VCF exists:", phase_filtered_path)

In [ ]:
# Split unfiltered and filtered VCFs by sample (one VCF per sample for HiPhase)
def phase_sample_vcf_path(sample: str, filtered: bool) -> str:
    tag = "filtered" if filtered else "unfiltered"
    return os.path.join(PHASE_DIR, f"phasing_{tag}_{PHASE_CHROM}_{PHASE_START//1e6:.0f}M_{PHASE_END//1e6:.0f}M_{sample}.vcf.gz")

for sample in PHASE_SAMPLES:
    for is_filtered, multi_path in [(False, phase_norm_path), (True, phase_filtered_path)]:
        out_path = phase_sample_vcf_path(sample, is_filtered)
        if not os.path.exists(out_path):
            subprocess.run(["bcftools", "view", "-s", sample, "-Oz", "-o", out_path, multi_path], check=True)
            subprocess.run(["tabix", "-p", "vcf", out_path], check=True)
print("Per-sample VCFs in", PHASE_DIR, "for", len(PHASE_SAMPLES), "samples (unfiltered + filtered)")

In [ ]:
# Stream only chr20:40-50 Mbp from GCS BAMs (no full download); index for HiPhase
# Ensure samtools on PATH (e.g. from same htslib as bcftools, or system)
if not shutil.which("samtools"):
    raise RuntimeError("samtools not found; install or add to PATH for BAM streaming")
bam_dir = os.path.join(PHASE_DIR, "bams")
for sample in PHASE_SAMPLES:
    for depth, bam_dict in [("15x", bams_15x), ("30x", bams_30x)]:
        gcs_bam = bam_dict.get(sample)
        if not gcs_bam:
            continue
        local_bam = os.path.join(bam_dir, f"{sample}_{depth}_chr20_40M_50M.bam")
        if not os.path.exists(local_bam):
            subprocess.run(
                ["samtools", "view", "-b", gcs_bam, PHASE_REGION, "-o", local_bam],
                check=True, env={**os.environ}
            )
        if not os.path.exists(local_bam + ".bai"):
            subprocess.run(["samtools", "index", local_bam], check=True)
print("Regional BAMs in", bam_dir)

In [ ]:
# Run HiPhase for each sample × unfiltered/filtered × 15x/30x
HIPHASE_THREADS = 4
for sample in PHASE_SAMPLES:
    for vcf_type, is_filtered in [("unfiltered", False), ("filtered", True)]:
        vcf_path = phase_sample_vcf_path(sample, is_filtered)
        for depth in ["15x", "30x"]:
            local_bam = os.path.join(PHASE_DIR, "bams", f"{sample}_{depth}_chr20_40M_50M.bam")
            if not os.path.exists(local_bam):
                continue
            out_vcf = os.path.join(PHASE_DIR, f"phased_{vcf_type}_{sample}_{depth}.vcf.gz")
            summary_tsv = os.path.join(PHASE_DIR, f"hiphase_summary_{vcf_type}_{sample}_{depth}.tsv")
            blocks_tsv = os.path.join(PHASE_DIR, f"hiphase_blocks_{vcf_type}_{sample}_{depth}.tsv")
            if os.path.exists(out_vcf):
                print("Skip (exists):", out_vcf)
                continue
            cmd = [
                "hiphase", "--disable-global-realignment",
                "--bam", local_bam, "--vcf", vcf_path,
                "--output-vcf", out_vcf, "--reference", ref_fa,
                "--threads", str(HIPHASE_THREADS),
                "--summary-file", summary_tsv, "--blocks-file", blocks_tsv,
            ]
            print("Running:", " ".join(cmd[:8]), "...")
            subprocess.run(cmd, check=True, env={**os.environ})
print("HiPhase runs done.")

In [ ]:
# Summarize phasing results: load HiPhase summary/block TSVs and compare unfiltered vs filtered, 15x vs 30x
import glob
summary_files = sorted(glob.glob(os.path.join(PHASE_DIR, "hiphase_summary_*.tsv")))
rows = []
for path in summary_files:
    base = os.path.basename(path)
    if not base.startswith("hiphase_summary_") or not base.endswith(".tsv"):
        continue
    parts = base.replace("hiphase_summary_", "").replace(".tsv", "").split("_")
    if len(parts) >= 3:
        vcf_type = parts[0]
        sample = parts[1]
        depth = "_".join(parts[2:])
    else:
        continue
    try:
        df = pd.read_csv(path, sep="\t")
        n_blocks = len(df)
        mean_len = df["block_length"].mean() if not df.empty and "block_length" in df.columns else None
        median_len = df["block_length"].median() if not df.empty and "block_length" in df.columns else None
        rows.append({"sample": sample, "vcf_type": vcf_type, "depth": depth, "n_blocks": n_blocks, "mean_block_len": mean_len, "median_block_len": median_len})
    except Exception as e:
        rows.append({"sample": sample, "vcf_type": vcf_type, "depth": depth, "n_blocks": None, "mean_block_len": None, "median_block_len": None, "error": str(e)})
if rows:
    summary_df = pd.DataFrame(rows)
    pd.set_option("display.max_columns", None)
    print(summary_df.to_string(index=False))
else:
    print("No hiphase_summary_*.tsv files found in", PHASE_DIR, "- run HiPhase cell first.")

In [ ]:
!wget -O hiphase-v1.6.0-x86_64-unknown-linux-gnu.tar.gz "https://github.com/PacificBiosciences/HiPhase/releases/download/v1.6.0/hiphase-v1.6.0-x86_64-unknown-linux-gnu.tar.gz"
!gunzip -c hiphase-v1.6.0-x86_64-unknown-linux-gnu.tar.gz | tar xvf -
!cp hiphase-v1.6.0-x86_64-unknown-linux-gnu/hiphase $HOME/.local/bin/
!hiphase

In [ ]:
bams_15x = {
    'HG00097': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00097_15x.bam',
    'HG00099': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00099_15x.bam',
    'HG00126': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00126_15x.bam',
    'HG00128': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00128_15x.bam',
    'HG00133': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00133_15x.bam',
    'HG00140': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00140_15x.bam',
    'HG00146': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00146_15x.bam',
    'HG00232': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00232_15x.bam',
    'HG00235': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00235_15x.bam',
    'HG00253': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00253_15x.bam',
    'HG00272': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00272_15x.bam',
    'HG00280': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00280_15x.bam',
    'HG00290': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00290_15x.bam',
    'HG002': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG002_15x.bam',
    'HG00320': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00320_15x.bam',
    'HG00321': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00321_15x.bam',
    'HG00323': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00323_15x.bam',
    'HG00329': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00329_15x.bam',
    'HG00344': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00344_15x.bam',
    'HG00350': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00350_15x.bam',
    'HG00408': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00408_15x.bam',
    'HG00423': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00423_15x.bam',
    'HG00438': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00438_15x.bam',
    'HG00544': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00544_15x.bam',
    'HG00558': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00558_15x.bam',
    'HG00597': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00597_15x.bam',
    'HG005': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG005_15x.bam',
    'HG00609': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00609_15x.bam',
    'HG00621': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00621_15x.bam',
    'HG00639': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00639_15x.bam',
    'HG00642': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00642_15x.bam',
    'HG00658': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00658_15x.bam',
    'HG00673': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00673_15x.bam',
    'HG00706': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00706_15x.bam',
    'HG00733': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00733_15x.bam',
    'HG00735': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00735_15x.bam',
    'HG00738': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00738_15x.bam',
    'HG00741': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00741_15x.bam',
    'HG01071': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01071_15x.bam',
    'HG01074': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01074_15x.bam',
    'HG01081': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01081_15x.bam',
    'HG01099': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01099_15x.bam',
    'HG01106': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01106_15x.bam',
    'HG01109': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01109_15x.bam',
    'HG01123': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01123_15x.bam',
    'HG01150': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01150_15x.bam',
    'HG01167': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01167_15x.bam',
    'HG01175': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01175_15x.bam',
    'HG01192': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01192_15x.bam',
    'HG01243': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01243_15x.bam',
    'HG01252': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01252_15x.bam',
    'HG01255': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01255_15x.bam',
    'HG01258': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01258_15x.bam',
    'HG01261': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01261_15x.bam',
    'HG01346': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01346_15x.bam',
    'HG01358': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01358_15x.bam',
    'HG01361': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01361_15x.bam',
    'HG01433': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01433_15x.bam',
    'HG01496': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01496_15x.bam',
    'HG01530': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01530_15x.bam',
    'HG01784': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01784_15x.bam',
    'HG01786': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01786_15x.bam',
    'HG01884': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01884_15x.bam',
    'HG01891': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01891_15x.bam',
    'HG01928': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01928_15x.bam',
    'HG01934': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01934_15x.bam',
    'HG01940': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01940_15x.bam',
    'HG01943': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01943_15x.bam',
    'HG01952': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01952_15x.bam',
    'HG01960': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01960_15x.bam',
    'HG01969': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01969_15x.bam',
    'HG01975': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01975_15x.bam',
    'HG01978': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01978_15x.bam',
    'HG01981': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01981_15x.bam',
    'HG01993': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01993_15x.bam',
    'HG02004': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02004_15x.bam',
    'HG02015': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02015_15x.bam',
    'HG02027': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02027_15x.bam',
    'HG02040': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02040_15x.bam',
    'HG02055': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02055_15x.bam',
    'HG02056': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02056_15x.bam',
    'HG02071': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02071_15x.bam',
    'HG02074': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02074_15x.bam',
    'HG02080': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02080_15x.bam',
    'HG02083': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02083_15x.bam',
    'HG02109': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02109_15x.bam',
    'HG02129': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02129_15x.bam',
    'HG02132': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02132_15x.bam',
    'HG02135': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02135_15x.bam',
    'HG02145': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02145_15x.bam',
    'HG02148': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02148_15x.bam',
    'HG02155': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02155_15x.bam',
    'HG02165': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02165_15x.bam',
    'HG02178': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02178_15x.bam',
    'HG02257': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02257_15x.bam',
    'HG02258': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02258_15x.bam',
    'HG02273': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02273_15x.bam',
    'HG02280': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02280_15x.bam',
    'HG02293': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02293_15x.bam',
    'HG02300': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02300_15x.bam',
    'HG02391': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02391_15x.bam',
    'HG02392': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02392_15x.bam',
    'HG02451': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02451_15x.bam',
    'HG02486': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02486_15x.bam',
    'HG02514': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02514_15x.bam',
    'HG02523': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02523_15x.bam',
    'HG02559': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02559_15x.bam',
    'HG02572': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02572_15x.bam',
    'HG02583': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02583_15x.bam',
    'HG02602': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02602_15x.bam',
    'HG02615': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02615_15x.bam',
    'HG02622': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02622_15x.bam',
    'HG02630': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02630_15x.bam',
    'HG02647': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02647_15x.bam',
    'HG02668': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02668_15x.bam',
    'HG02698': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02698_15x.bam',
    'HG02717': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02717_15x.bam',
    'HG02723': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02723_15x.bam',
    'HG02735': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02735_15x.bam',
    'HG02738': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02738_15x.bam',
    'HG02809': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02809_15x.bam',
    'HG02818': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02818_15x.bam',
    'HG02841': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02841_15x.bam',
    'HG02886': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02886_15x.bam',
    'HG02922': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02922_15x.bam',
    'HG02965': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02965_15x.bam',
    'HG02976': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02976_15x.bam',
    'HG02984': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02984_15x.bam',
    'HG03017': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03017_15x.bam',
    'HG03041': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03041_15x.bam',
    'HG03050': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03050_15x.bam',
    'HG03098': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03098_15x.bam',
    'HG03130': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03130_15x.bam',
    'HG03139': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03139_15x.bam',
    'HG03195': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03195_15x.bam',
    'HG03209': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03209_15x.bam',
    'HG03225': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03225_15x.bam',
    'HG03239': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03239_15x.bam',
    'HG03270': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03270_15x.bam',
    'HG03369': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03369_15x.bam',
    'HG03453': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03453_15x.bam',
    'HG03470': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03470_15x.bam',
    'HG03471': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03471_15x.bam',
    'HG03486': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03486_15x.bam',
    'HG03492': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03492_15x.bam',
    'HG03516': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03516_15x.bam',
    'HG03521': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03521_15x.bam',
    'HG03540': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03540_15x.bam',
    'HG03579': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03579_15x.bam',
    'HG03583': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03583_15x.bam',
    'HG03654': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03654_15x.bam',
    'HG03669': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03669_15x.bam',
    'HG03688': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03688_15x.bam',
    'HG03704': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03704_15x.bam',
    'HG03710': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03710_15x.bam',
    'HG03742': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03742_15x.bam',
    'HG03784': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03784_15x.bam',
    'HG03804': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03804_15x.bam',
    'HG03816': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03816_15x.bam',
    'HG03831': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03831_15x.bam',
    'HG03834': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03834_15x.bam',
    'HG03874': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03874_15x.bam',
    'HG03927': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03927_15x.bam',
    'HG03942': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03942_15x.bam',
    'HG04115': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04115_15x.bam',
    'HG04157': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04157_15x.bam',
    'HG04160': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04160_15x.bam',
    'HG04184': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04184_15x.bam',
    'HG04187': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04187_15x.bam',
    'HG04199': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04199_15x.bam',
    'HG04204': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04204_15x.bam',
    'HG04228': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04228_15x.bam',
    'HG06807': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG06807_15x.bam',
    'NA18505': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18505_15x.bam',
    'NA18508': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18508_15x.bam',
    'NA18522': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18522_15x.bam',
    'NA18565': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18565_15x.bam',
    'NA18570': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18570_15x.bam',
    'NA18608': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18608_15x.bam',
    'NA18620': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18620_15x.bam',
    'NA18747': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18747_15x.bam',
    'NA18879': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18879_15x.bam',
    'NA18906': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18906_15x.bam',
    'NA18940': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18940_15x.bam',
    'NA18943': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18943_15x.bam',
    'NA18944': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18944_15x.bam',
    'NA18945': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18945_15x.bam',
    'NA18948': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18948_15x.bam',
    'NA18952': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18952_15x.bam',
    'NA18959': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18959_15x.bam',
    'NA18960': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18960_15x.bam',
    'NA18967': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18967_15x.bam',
    'NA18970': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18970_15x.bam',
    'NA18971': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18971_15x.bam',
    'NA18974': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18974_15x.bam',
    'NA18976': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18976_15x.bam',
    'NA18982': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18982_15x.bam',
    'NA18983': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18983_15x.bam',
    'NA19036': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19036_15x.bam',
    'NA19043': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19043_15x.bam',
    'NA19087': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19087_15x.bam',
    'NA19159': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19159_15x.bam',
    'NA19185': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19185_15x.bam',
    'NA19240': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19240_15x.bam',
    'NA19338': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19338_15x.bam',
    'NA19391': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19391_15x.bam',
    'NA19443': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19443_15x.bam',
    'NA19468': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19468_15x.bam',
    'NA19682': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19682_15x.bam',
    'NA19700': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19700_15x.bam',
    'NA19776': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19776_15x.bam',
    'NA19835': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19835_15x.bam',
    'NA19909': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19909_15x.bam',
    'NA20129': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20129_15x.bam',
    'NA20282': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20282_15x.bam',
    'NA20346': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20346_15x.bam',
    'NA20503': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20503_15x.bam',
    'NA20752': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20752_15x.bam',
    'NA20762': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20762_15x.bam',
    'NA20799': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20799_15x.bam',
    'NA20805': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20805_15x.bam',
    'NA20806': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20806_15x.bam',
    'NA20809': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20809_15x.bam',
    'NA20827': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20827_15x.bam',
    'NA20850': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20850_15x.bam',
    'NA20870': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20870_15x.bam',
    'NA20905': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20905_15x.bam',
    'NA21093': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21093_15x.bam',
    'NA21102': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21102_15x.bam',
    'NA21106': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21106_15x.bam',
    'NA21110': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21110_15x.bam',
    'NA21144': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21144_15x.bam',
    'NA21309': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21309_15x.bam',
}

In [ ]:
bams_30x = {
    'HG00097': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00097_30x.bam',
    'HG00099': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00099_30x.bam',
    'HG00126': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00126_30x.bam',
    'HG00128': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00128_30x.bam',
    'HG00133': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00133_30x.bam',
    'HG00140': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00140_30x.bam',
    'HG00146': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00146_30x.bam',
    'HG00232': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00232_30x.bam',
    'HG00235': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00235_30x.bam',
    'HG00253': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00253_30x.bam',
    'HG00272': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00272_30x.bam',
    'HG00280': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00280_30x.bam',
    'HG00290': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00290_30x.bam',
    'HG002': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG002_30x.bam',
    'HG00320': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00320_30x.bam',
    'HG00321': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00321_30x.bam',
    'HG00323': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00323_30x.bam',
    'HG00329': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00329_30x.bam',
    'HG00344': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00344_30x.bam',
    'HG00350': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00350_30x.bam',
    'HG00408': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00408_30x.bam',
    'HG00423': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00423_30x.bam',
    'HG00438': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00438_30x.bam',
    'HG00544': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00544_30x.bam',
    'HG00558': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00558_30x.bam',
    'HG00597': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00597_30x.bam',
    'HG005': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG005_30x.bam',
    'HG00609': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00609_30x.bam',
    'HG00621': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00621_30x.bam',
    'HG00639': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00639_30x.bam',
    'HG00642': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00642_30x.bam',
    'HG00658': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00658_30x.bam',
    'HG00673': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00673_30x.bam',
    'HG00706': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00706_30x.bam',
    'HG00733': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00733_30x.bam',
    'HG00735': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00735_30x.bam',
    'HG00738': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00738_30x.bam',
    'HG00741': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG00741_30x.bam',
    'HG01071': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01071_30x.bam',
    'HG01074': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01074_30x.bam',
    'HG01081': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01081_30x.bam',
    'HG01099': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01099_30x.bam',
    'HG01106': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01106_30x.bam',
    'HG01109': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01109_30x.bam',
    'HG01123': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01123_30x.bam',
    'HG01150': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01150_30x.bam',
    'HG01167': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01167_30x.bam',
    'HG01175': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01175_30x.bam',
    'HG01192': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01192_30x.bam',
    'HG01243': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01243_30x.bam',
    'HG01252': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01252_30x.bam',
    'HG01255': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01255_30x.bam',
    'HG01258': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01258_30x.bam',
    'HG01261': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01261_30x.bam',
    'HG01346': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01346_30x.bam',
    'HG01358': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01358_30x.bam',
    'HG01361': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01361_30x.bam',
    'HG01433': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01433_30x.bam',
    'HG01496': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01496_30x.bam',
    'HG01530': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01530_30x.bam',
    'HG01784': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01784_30x.bam',
    'HG01786': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01786_30x.bam',
    'HG01884': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01884_30x.bam',
    'HG01891': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01891_30x.bam',
    'HG01928': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01928_30x.bam',
    'HG01934': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01934_30x.bam',
    'HG01940': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01940_30x.bam',
    'HG01943': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01943_30x.bam',
    'HG01952': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01952_30x.bam',
    'HG01960': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01960_30x.bam',
    'HG01969': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01969_30x.bam',
    'HG01975': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01975_30x.bam',
    'HG01978': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01978_30x.bam',
    'HG01981': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01981_30x.bam',
    'HG01993': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG01993_30x.bam',
    'HG02004': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02004_30x.bam',
    'HG02015': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02015_30x.bam',
    'HG02027': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02027_30x.bam',
    'HG02040': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02040_30x.bam',
    'HG02055': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02055_30x.bam',
    'HG02056': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02056_30x.bam',
    'HG02071': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02071_30x.bam',
    'HG02074': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02074_30x.bam',
    'HG02080': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02080_30x.bam',
    'HG02083': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02083_30x.bam',
    'HG02109': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02109_30x.bam',
    'HG02129': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02129_30x.bam',
    'HG02132': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02132_30x.bam',
    'HG02135': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02135_30x.bam',
    'HG02145': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02145_30x.bam',
    'HG02148': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02148_30x.bam',
    'HG02155': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02155_30x.bam',
    'HG02165': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02165_30x.bam',
    'HG02178': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02178_30x.bam',
    'HG02257': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02257_30x.bam',
    'HG02258': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02258_30x.bam',
    'HG02273': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02273_30x.bam',
    'HG02280': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02280_30x.bam',
    'HG02293': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02293_30x.bam',
    'HG02300': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02300_30x.bam',
    'HG02391': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02391_30x.bam',
    'HG02392': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02392_30x.bam',
    'HG02451': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02451_30x.bam',
    'HG02486': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02486_30x.bam',
    'HG02514': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02514_30x.bam',
    'HG02523': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02523_30x.bam',
    'HG02559': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02559_30x.bam',
    'HG02572': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02572_30x.bam',
    'HG02583': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02583_30x.bam',
    'HG02602': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02602_30x.bam',
    'HG02615': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02615_30x.bam',
    'HG02622': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02622_30x.bam',
    'HG02630': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02630_30x.bam',
    'HG02647': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02647_30x.bam',
    'HG02668': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02668_30x.bam',
    'HG02698': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02698_30x.bam',
    'HG02717': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02717_30x.bam',
    'HG02723': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02723_30x.bam',
    'HG02735': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02735_30x.bam',
    'HG02738': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02738_30x.bam',
    'HG02809': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02809_30x.bam',
    'HG02818': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02818_30x.bam',
    'HG02841': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02841_30x.bam',
    'HG02886': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02886_30x.bam',
    'HG02922': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02922_30x.bam',
    'HG02965': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02965_30x.bam',
    'HG02976': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02976_30x.bam',
    'HG02984': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG02984_30x.bam',
    'HG03017': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03017_30x.bam',
    'HG03041': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03041_30x.bam',
    'HG03050': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03050_30x.bam',
    'HG03098': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03098_30x.bam',
    'HG03130': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03130_30x.bam',
    'HG03139': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03139_30x.bam',
    'HG03195': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03195_30x.bam',
    'HG03209': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03209_30x.bam',
    'HG03225': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03225_30x.bam',
    'HG03239': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03239_30x.bam',
    'HG03270': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03270_30x.bam',
    'HG03369': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03369_30x.bam',
    'HG03453': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03453_30x.bam',
    'HG03470': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03470_30x.bam',
    'HG03471': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03471_30x.bam',
    'HG03486': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03486_30x.bam',
    'HG03492': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03492_30x.bam',
    'HG03516': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03516_30x.bam',
    'HG03521': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03521_30x.bam',
    'HG03540': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03540_30x.bam',
    'HG03579': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03579_30x.bam',
    'HG03583': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03583_30x.bam',
    'HG03654': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03654_30x.bam',
    'HG03669': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03669_30x.bam',
    'HG03688': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03688_30x.bam',
    'HG03704': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03704_30x.bam',
    'HG03710': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03710_30x.bam',
    'HG03742': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03742_30x.bam',
    'HG03784': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03784_30x.bam',
    'HG03804': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03804_30x.bam',
    'HG03816': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03816_30x.bam',
    'HG03831': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03831_30x.bam',
    'HG03834': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03834_30x.bam',
    'HG03874': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03874_30x.bam',
    'HG03927': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03927_30x.bam',
    'HG03942': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG03942_30x.bam',
    'HG04115': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04115_30x.bam',
    'HG04157': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04157_30x.bam',
    'HG04160': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04160_30x.bam',
    'HG04184': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04184_30x.bam',
    'HG04187': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04187_30x.bam',
    'HG04199': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04199_30x.bam',
    'HG04204': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04204_30x.bam',
    'HG04228': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG04228_30x.bam',
    'HG06807': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/HG06807_30x.bam',
    'NA18505': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18505_30x.bam',
    'NA18508': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18508_30x.bam',
    'NA18522': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18522_30x.bam',
    'NA18565': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18565_30x.bam',
    'NA18570': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18570_30x.bam',
    'NA18608': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18608_30x.bam',
    'NA18620': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18620_30x.bam',
    'NA18747': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18747_30x.bam',
    'NA18879': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18879_30x.bam',
    'NA18906': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18906_30x.bam',
    'NA18940': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18940_30x.bam',
    'NA18943': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18943_30x.bam',
    'NA18944': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18944_30x.bam',
    'NA18945': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18945_30x.bam',
    'NA18948': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18948_30x.bam',
    'NA18952': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18952_30x.bam',
    'NA18959': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18959_30x.bam',
    'NA18960': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18960_30x.bam',
    'NA18967': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18967_30x.bam',
    'NA18970': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18970_30x.bam',
    'NA18971': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18971_30x.bam',
    'NA18974': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18974_30x.bam',
    'NA18976': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18976_30x.bam',
    'NA18982': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18982_30x.bam',
    'NA18983': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA18983_30x.bam',
    'NA19036': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19036_30x.bam',
    'NA19043': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19043_30x.bam',
    'NA19087': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19087_30x.bam',
    'NA19159': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19159_30x.bam',
    'NA19185': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19185_30x.bam',
    'NA19240': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19240_30x.bam',
    'NA19338': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19338_30x.bam',
    'NA19391': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19391_30x.bam',
    'NA19443': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19443_30x.bam',
    'NA19468': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19468_30x.bam',
    'NA19682': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19682_30x.bam',
    'NA19700': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19700_30x.bam',
    'NA19776': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19776_30x.bam',
    'NA19835': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19835_30x.bam',
    'NA19909': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA19909_30x.bam',
    'NA20129': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20129_30x.bam',
    'NA20282': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20282_30x.bam',
    'NA20346': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20346_30x.bam',
    'NA20503': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20503_30x.bam',
    'NA20752': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20752_30x.bam',
    'NA20762': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20762_30x.bam',
    'NA20799': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20799_30x.bam',
    'NA20805': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20805_30x.bam',
    'NA20806': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20806_30x.bam',
    'NA20809': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20809_30x.bam',
    'NA20827': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20827_30x.bam',
    'NA20850': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20850_30x.bam',
    'NA20870': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20870_30x.bam',
    'NA20905': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA20905_30x.bam',
    'NA21093': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21093_30x.bam',
    'NA21102': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21102_30x.bam',
    'NA21106': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21106_30x.bam',
    'NA21110': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21110_30x.bam',
    'NA21144': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21144_30x.bam',
    'NA21309': 'gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/PacBio_hifi/subsampled/NA21309_30x.bam',
}